# Notebook 4 — Predictive Model (XGBoost)

**Project:** Robson Classification and C-Section Predictive System
**Stage:** Phase 3 — Building and validating the predictive model
**Scripts covered (8):**

| Script | Role |
|--------|------|
| `step4_5_clean.py` | Sentinel values → NaN, no imputation |
| `step6_encode.py` | RF_ARTEC conditionality fix, ILLB_R separation, one-hot encoding |
| `step7_9_split.py` | Narrow vs Wide population, stratified 60/20/20 split, imbalance check |
| `step10_train.py` | First training attempt (WITH class-weighting) — **the old version** |
| `step10_train_v2.py` | Memory-safe rewrite — still uses class-weighting |
| `step11b_calibration.py` | **THE KEY CORRECTION** — discovers class-weighting is the problem |
| `step12_threshold.py` | Threshold analysis — F2 commitment withdrawn |
| `step13_evaluate.py` | Final test-set evaluation — opened once, never re-run |

---

### The story arc of this notebook

This notebook documents a **mistake, its discovery, and its correction** — which the supervisor identified as a strength, not a weakness:

1. **Steps 4-9:** Clean, encode, split — straightforward preparation
2. **Step 10:** Train XGBoost with `scale_pos_weight` to handle class imbalance
3. **Step 11b:** Discover that `scale_pos_weight=7.6` inflates every predicted probability, corrupting calibration (Brier 0.1857)
4. **The fix:** Remove class-weighting → Brier drops to **0.0886** with no loss in discrimination
5. **Steps 12-13:** Threshold analysis and final evaluation on held-out test data

### Why calibration matters more than discrimination here

Most clinical prediction papers care about ranking (who is at higher risk). This project does something different: it **sums predicted probabilities** to produce an expected CS rate, then subtracts from observed to estimate excess. A model whose probabilities are inflated gives an inflated expected rate and an **understated gap**. The numbers themselves have to be right, not just the ranking.

### The corrected numbers (validation set, during development)

| Metric | Old (with weighting) | **Corrected (without)** |
|--------|---------------------|------------------------|
| Brier score | 0.1857 | **0.0886** |
| ROC-AUC | 0.7985 | **0.7985** (unchanged) |
| Mean predicted | ~39.4% | **11.70%** (matches 11.72% observed) |

Final test-set ROC-AUC: **0.7991** (opened once, 486,820 births). The +0.0006 difference from validation confirms a clean split.

### A fundamental distinction

**Robson** = rule-based classification. Deterministic. No training.
**XGBoost** = predictive model. Learns from data. Has accuracy to measure.

The two are complementary tools, not alternatives. The model is NOT in the critical path of the Egypt gap estimate — that uses observed group-specific rates from the Robson table. The model's role is **explanatory** (SHAP) and **demonstrating** that clinical variables account for part of the variation but not all of it.

---

## Step 4+5 — Load and Clean

**Script:** `step4_5_clean.py`

### The problem this solves

CDC does not leave unknown values blank. It records them as numbers:

| Field | Sentinel | What it looks like |
|-------|----------|-------------------|
| Gestational age | 99 | A 99-week pregnancy |
| Birth weight | 9999 | A 10-kg baby |
| BMI | 99.9 | BMI of 100 |
| Height | 99 | 99 inches (2.5 meters) |
| Previous CS count | 99 | 99 prior caesareans |

Nothing raises an error. The value flows into every mean, split, and SHAP plot as though real. This is the **largest silent data-quality risk** in this source.

### The approach

- Each sentinel → genuine `NaN`
- Every conversion is printed and auditable
- **Nothing is imputed.** XGBoost handles missingness natively by treating it as a branch during splitting — preserving the information that a value was absent instead of inventing one
- Output: `model_data.parquet` — 24 model columns + target + Robson group

In [ ]:
"""
=============================================================================
STEP 4 + 5 — LOAD AND CLEAN
=============================================================================
    python step4_5_clean.py

THE PROBLEM THIS SOLVES

CDC does not leave unknown values blank. It records them as numbers:

    gestational age unknown ....... 99   (weeks)
    birth weight unknown .......... 9999 (grams)
    BMI unknown ................... 99.9
    Apgar unknown ................. 99
    previous caesareans unknown ... 99
    height unknown ................ 99   (inches)

Nothing raises an error. The value simply flows into every mean, every
split and every SHAP plot as though a woman were 99 weeks pregnant. It is
the largest silent data-quality risk in this source, and it is invisible
unless you go looking for it.

This script converts each sentinel to a genuine missing value, and prints
how many it found in each column so the decision is auditable rather than
assumed. Nothing is imputed: XGBoost handles missingness natively by
treating it as a branch during splitting, which preserves the information
that a value was absent instead of inventing one.

Output: model_data.parquet — the 24 model columns plus the target, the
Robson group (for later evaluation splits) and the classification status.
=============================================================================
"""

import pandas as pd
import numpy as np
from pathlib import Path
import sys

# ---------------------------------------------------------------------------
# Feature list, inlined so this runs as a single file.
# Generated from data_dictionary.py — 24 of 59 columns.
# The test for inclusion: was this fact known BEFORE the delivery decision?
# ---------------------------------------------------------------------------
TARGET = "is_cesarean"

MODEL_FEATURES = [
    # maternal characteristics
    "MAGER",        # mother's age (years)
    "M_Ht_In",      # height (inches)
    "PWgt_R",       # pre-pregnancy weight (lb)
    "WTGAIN",       # weight gain in pregnancy (lb)
    "BMI",          # body mass index
    "MEDUC",        # education (ordinal)
    "DMAR",         # marital status
    "CIG_REC",      # smoked during pregnancy
    "PRECARE5",     # month antenatal care began (ordinal)
    # obstetric history
    "PRIORLIVE",    # previous live births
    "PRIORDEAD",    # previous stillbirths
    "RF_CESARN",    # number of previous caesareans
    "ILLB_R",       # interval since last live birth
    "RF_PPTERM",    # previous preterm birth
    # comorbidity
    "RF_PDIAB",     # pre-pregnancy diabetes
    "RF_GDIAB",     # gestational diabetes
    "RF_PHYPE",     # chronic hypertension
    "RF_GHYPE",     # gestational hypertension
    "RF_EHYPE",     # eclampsia
    "RF_INFTR",     # infertility treatment
    "RF_ARTEC",     # assisted reproductive technology
    # current pregnancy
    "ME_PRES",      # fetal presentation (nominal)
    "DPLURAL",      # number of fetuses
    "OEGest_Comb",  # gestational age (weeks)
]

# sentinel -> the codes that mean "unknown" in that column
SENTINELS = {
    "MAGER":       [],                 # 12-50, no sentinel
    "M_Ht_In":     [99],
    "PWgt_R":      [999],
    "WTGAIN":      [99],
    "BMI":         [99.9],
    "MEDUC":       [9],
    "DMAR":        [9],
    "PRIORLIVE":   [99],
    "PRIORDEAD":   [99],
    "RF_CESARN":   [99],
    "ILLB_R":      [888, 999],         # 888 = not applicable (first birth)
    "OEGest_Comb": [99],
    "DPLURAL":     [9],
}
# Y/N/U fields -> True / False / missing
YN_COLS = ["CIG_REC", "RF_PPTERM", "RF_PDIAB", "RF_GDIAB", "RF_PHYPE",
           "RF_GHYPE", "RF_EHYPE", "RF_INFTR", "RF_ARTEC"]
# categorical codes kept as labels, with their unknown code
CAT_UNKNOWN = {"ME_PRES": ["9"], "PRECARE5": ["5"]}

SRC = next((f for f in ["natality_2023_extended.parquet",
                        "natality_2023_filtered.parquet"] if Path(f).exists()), None)
if SRC is None:
    raise SystemExit("No parquet cache found. Run robson_analysis.py first.")

print(f"Loading {SRC} ...")
keep = MODEL_FEATURES + [TARGET, "Robson_group", "Robson_group_sort_order",
                         "Classification_status"]
df = pd.read_parquet(SRC)
missing_cols = [c for c in keep if c not in df.columns]
if missing_cols:
    raise SystemExit(f"Columns not in the cache: {missing_cols}")
d = df[keep].copy()
print(f"  {len(d):,} rows, {len(MODEL_FEATURES)} model columns\n")

print("=" * 74)
print("CLEANING — what was found and removed")
print("=" * 74)
print(f"{'column':<14} {'type':<12} {'sentinel':>12} {'flagged':>12} {'% missing':>11}")
print("-" * 74)

report = []
for col in MODEL_FEATURES:
    if col in YN_COLS:
        s = d[col].astype(str).str.strip().str.upper()
        out = pd.Series(np.nan, index=d.index, dtype="object")
        out[s == "Y"] = 1.0
        out[s == "N"] = 0.0
        n_unknown = int((~s.isin(["Y", "N"])).sum())
        d[col] = pd.to_numeric(out, errors="coerce")
        kind, sent = "binary Y/N", "U"
    elif col in CAT_UNKNOWN:
        s = d[col].astype(str).str.strip()
        n_unknown = int(s.isin(CAT_UNKNOWN[col]).sum())
        d[col] = s.where(~s.isin(CAT_UNKNOWN[col]), np.nan)
        kind, sent = "categorical", ",".join(CAT_UNKNOWN[col])
    else:
        x = pd.to_numeric(d[col], errors="coerce")
        codes = SENTINELS.get(col, [])
        n_unknown = int(x.isin(codes).sum()) if codes else 0
        if codes:
            x = x.where(~x.isin(codes))
        d[col] = x
        kind = "numeric"
        sent = ",".join(str(c) for c in codes) if codes else "none"

    pct = 100 * d[col].isna().mean()
    report.append((col, kind, sent, n_unknown, pct))
    print(f"{col:<14} {kind:<12} {sent:>12} {n_unknown:>12,} {pct:>10.2f}%")

rep = pd.DataFrame(report, columns=["column","type","sentinel","n_flagged","pct_missing"])
rep.to_csv("cleaning_report.csv", index=False)

print("\n" + "=" * 74)
print("SANITY CHECK — ranges after cleaning")
print("=" * 74)
num = [c for c in MODEL_FEATURES if c not in CAT_UNKNOWN]
desc = d[num].describe().T[["min", "max", "mean"]].round(2)
print(desc.to_string())
print("""
  Read the max column. If any of these still shows 99, 999 or 9999 where
  that value is clinically impossible, a sentinel has been missed and must
  be added to SENTINELS above before going further.
""")

flagged = []
for col in num:
    mx = d[col].max()
    if col in ("OEGest_Comb",) and mx > 50: flagged.append(col)
    if col in ("M_Ht_In",) and mx > 90: flagged.append(col)
    if col in ("RF_CESARN","PRIORLIVE","PRIORDEAD") and mx > 30: flagged.append(col)
    if col == "BMI" and mx > 90: flagged.append(col)
if flagged:
    print(f"  WARNING — implausible maximum still present in: {flagged}")
else:
    print("  All ranges plausible. No sentinel left behind.")

print("\n" + "=" * 74)
print("MISSINGNESS SUMMARY")
print("=" * 74)
worst = rep.sort_values("pct_missing", ascending=False).head(8)
print(worst[["column","pct_missing"]].to_string(index=False))
print("""
  Nothing is imputed. XGBoost treats missingness as a branch when it splits,
  which keeps the fact that a value was absent as information rather than
  replacing it with a guess. A high missing rate is a limitation to report,
  not a hole to fill quietly.

  ILLB_R is expected to be largely missing: 888 means 'not applicable',
  which is the correct value for a first birth. That is a real absence, not
  a recording failure, and the distinction belongs in the write-up.
""")

d.to_parquet("model_data.parquet", index=False)
print(f"Saved model_data.parquet — {len(d):,} rows, {len(d.columns)} columns")
print("Saved cleaning_report.csv")
print("\nNext: step 6 — one-hot encoding.")

---

## Step 6 — Verify Conditional Fields, Then Encode

**Script:** `step6_encode.py`

### Two discoveries before encoding

**1. RF_ARTEC is conditional on RF_INFTR:**
The field was 97.61% missing with 77% "yes" among the remainder. ART is ~2% of US births, not 77%. Cross-tabulation confirmed: the field is only asked of women who had infertility treatment. "Missing" means "not applicable" → recoded to 0 (no ART). This is a factual recode, not imputation.

**2. ILLB_R mixes two absences:**
Code 888 = "not applicable" (first birth) vs 999 = genuinely unknown. Step 5 lumped them. A companion flag `ILLB_first_birth` restores the distinction.

### Encoding rules

- **Nominal** (ME_PRES) → one-hot: cephalic/breech/other as separate 0/1 columns
- **Ordinal** (MEDUC, PRECARE5, DPLURAL) → left numeric (the ordering is real information)
- Output: `model_ready.parquet` with the final feature set

In [ ]:
"""
=============================================================================
STEP 6 — VERIFY THE CONDITIONAL FIELDS, THEN ENCODE
=============================================================================
    python step6_encode.py

TWO THINGS BEFORE ENCODING

1. RF_ARTEC came out 97.61% missing with 77% "yes" among the remainder.
   Assisted reproduction is about 2% of US births, not 77%, so one of two
   things is true: the field position is wrong, or the field is CONDITIONAL
   — asked only of women who had infertility treatment. If it is
   conditional, "missing" means "not applicable", and a woman who had no
   infertility treatment did not have ART. Recording that as unknown throws
   away a fact we hold.

   This script cross-tabulates RF_ARTEC against RF_INFTR and decides from
   the data rather than from assumption. If the pattern confirms
   conditionality, it recodes; if it does not, it stops and says so.

2. ILLB_R mixes two different absences. 888 means "not applicable" — the
   correct value for a first birth — while 999 means genuinely unknown.
   Step 5 lumped them. They are separated here, because "she has no
   previous birth" and "we failed to record the interval" are different
   facts, and a model that cannot tell them apart is being lied to.

THEN ENCODING

Nominal columns become one-hot: a separate 0/1 column per category. Without
this, a numeric code tells the model that breech (2) is larger than
cephalic (1), which is meaningless and lets the tree split on an ordering
that does not exist.

Ordinal columns stay numeric. MEDUC and PRECARE5 have a real order — more
education is more, later antenatal booking is later — so the ordering is
information, not an artefact.
=============================================================================
"""

import pandas as pd
import numpy as np
from pathlib import Path

SRC = "model_data.parquet"
if not Path(SRC).exists():
    raise SystemExit(f"{SRC} not found. Run step4_5_clean.py first.")

d = pd.read_parquet(SRC)
print(f"Loaded {len(d):,} rows, {len(d.columns)} columns\n")

# =============================================================================
print("=" * 74)
print("1. IS RF_ARTEC CONDITIONAL ON RF_INFTR?")
print("=" * 74)

inf = d["RF_INFTR"]
art = d["RF_ARTEC"]
ct = pd.crosstab(inf.fillna(-1).map({1.0: "infertility treatment: YES",
                                     0.0: "infertility treatment: NO",
                                     -1: "infertility treatment: unknown"}),
                 art.isna().map({True: "ARTEC missing", False: "ARTEC recorded"}))
print(ct.to_string())

no_treat = d[inf == 0.0]
pct_missing_when_no = 100 * no_treat["RF_ARTEC"].isna().mean() if len(no_treat) else np.nan
yes_treat = d[inf == 1.0]
pct_missing_when_yes = 100 * yes_treat["RF_ARTEC"].isna().mean() if len(yes_treat) else np.nan

print(f"\n  ARTEC missing where there was NO infertility treatment : {pct_missing_when_no:.2f}%")
print(f"  ARTEC missing where there WAS infertility treatment    : {pct_missing_when_yes:.2f}%")

CONDITIONAL = pct_missing_when_no > 90 and pct_missing_when_yes < 25
if CONDITIONAL:
    print("""
  CONFIRMED conditional. The field is asked only of women who had
  infertility treatment, so "missing" means "not applicable".

  Recoding: no infertility treatment -> ART = 0. This is a factual recode,
  not imputation: a woman who had no fertility treatment did not conceive
  by assisted reproduction.
""")
    d.loc[inf == 0.0, "RF_ARTEC"] = 0.0
    print(f"  RF_ARTEC missing after recode: {100*d['RF_ARTEC'].isna().mean():.2f}%")
    print(f"  RF_ARTEC prevalence now      : {100*d['RF_ARTEC'].mean():.2f}%   "
          f"(published US ART share of births is roughly 2%)")
else:
    print("""
  NOT the expected conditional pattern. Do not recode. The field position
  or the code values need re-checking against the User Guide before
  RF_ARTEC is used at all. It is dropped from the feature set for now.
""")
    d = d.drop(columns=["RF_ARTEC"])

# =============================================================================
print("\n" + "=" * 74)
print("2. SEPARATING THE TWO KINDS OF ABSENCE IN ILLB_R")
print("=" * 74)
print("""
  Step 5 turned both 888 (not applicable — no previous birth) and 999
  (unknown) into the same missing value. PRIORLIVE already tells us which
  women have no previous birth, so a companion flag restores the
  distinction without re-reading the raw file.
""")
d["ILLB_first_birth"] = (pd.to_numeric(d["PRIORLIVE"], errors="coerce") == 0).astype(float)
print(f"  First births (no previous live birth): {100*d['ILLB_first_birth'].mean():.2f}%")
print(f"  ILLB_R missing                       : {100*d['ILLB_R'].isna().mean():.2f}%")
print("""
  The two should be close. Where ILLB_R is missing but this is NOT a first
  birth, the interval is genuinely unrecorded — that residual is the real
  missingness, and it is what belongs in the limitations.
""")
resid = ((d["ILLB_R"].isna()) & (d["ILLB_first_birth"] == 0)).mean()
print(f"  Genuinely unrecorded interval: {100*resid:.2f}% of all births")

# =============================================================================
print("\n" + "=" * 74)
print("3. ONE-HOT ENCODING")
print("=" * 74)

NOMINAL = ["ME_PRES"]          # cephalic / breech / other — no ordering
ORDINAL = ["MEDUC", "PRECARE5", "DPLURAL"]

print("  Nominal (one-hot):", NOMINAL)
print("  Ordinal (left numeric):", ORDINAL)
print("""
  ME_PRES is the only truly nominal clinical field in the set. Encoding it
  numerically would let a tree split on "presentation <= 1.5", which asserts
  an ordering between cephalic, breech and other that does not exist.
""")

for col in NOMINAL:
    s = d[col].astype(str).str.strip()
    labels = {"1": "cephalic", "2": "breech", "3": "other"}
    for code, name in labels.items():
        d[f"{col}_{name}"] = (s == code).astype(float)
        # keep genuine unknowns as missing rather than as three zeros
        d.loc[d[col].isna(), f"{col}_{name}"] = np.nan
    print(f"  {col} -> " + ", ".join(f"{col}_{n}" for n in labels.values()))
    d = d.drop(columns=[col])

# =============================================================================
FEATURES = [c for c in d.columns
            if c not in ["is_cesarean", "Robson_group", "Robson_group_sort_order",
                         "Classification_status"]]

print("\n" + "=" * 74)
print("FINAL FEATURE SET")
print("=" * 74)
print(f"  {len(FEATURES)} features\n")
for i, c in enumerate(FEATURES, 1):
    miss = 100 * d[c].isna().mean()
    print(f"  {i:>2}. {c:<22} {miss:>6.2f}% missing")

d.to_parquet("model_ready.parquet", index=False)
pd.Series(FEATURES).to_csv("feature_list.csv", index=False, header=["feature"])
print(f"\nSaved model_ready.parquet — {len(d):,} rows")
print("Saved feature_list.csv")
print("\nNext: step 7 — restrict the training population.")

---

## Steps 7, 8, 9 — Training Population, Split, and Imbalance

**Script:** `step7_9_split.py`

### Step 7: Which births the model learns from

Three exclusions are not open to argument:

| Group | Why excluded |
|-------|-------------|
| **2b** | Pre-labour CS by definition — the answer IS the classification |
| **4b** | Same |
| **9** | CDC "other" is contaminated (documented in Notebook 2) |

**The open question:** Should Groups 5, 6, 7, 8, 10 also be excluded? They are NOT leakage (a breech CAN deliver vaginally), but their rates are so high they may add noise rather than signal. Two populations are built:

- **NARROW:** Groups 1, 2a, 3, 4a only (where genuine uncertainty lives)
- **WIDE:** Everything except 2b, 4b, 9

Both are carried forward — the decision is made from evidence in Step 10.

### Step 8: Stratified 60/20/20 split

Stratified on the target so the CS proportion is identical in train/valid/test.

### Step 9: Imbalance measured, not assumed

The national rate is 32%, but after removing fixed-outcome groups, the narrow population runs at ~12%. The direction of imbalance cannot be predicted from the headline.

In [ ]:
"""
=============================================================================
STEPS 7, 8 and 9 — BUILD THE TRAINING SET
=============================================================================
    python step7_9_split.py

STEP 7 — WHICH BIRTHS THE MODEL LEARNS FROM

Three exclusions are not open to argument:

    2b  nulliparous, term, cephalic, PRE-LABOUR caesarean ..... 100% by definition
    4b  multiparous, same ..................................... 100% by definition
    9   abnormal lie .......................................... definitionally
                                                                contaminated in
                                                                this data source

Group membership in 2b and 4b is decided by the same fact the model is
asked to predict. A model trained on them reads the answer out of the
question and returns a high score that measures the classification, not
any clinical pattern. Group 9 is excluded for a different reason: CDC's
"other" presentation is broader than WHO's transverse lie, so the variable
itself is unreliable here.

The remaining question was left open on purpose, to be settled by evidence
rather than by preference: should the model also exclude groups 5, 6, 7, 8
and 10? Those are not leakage. A breech CAN deliver vaginally and a repeat
caesarean IS a decision — their high rates are clinical findings, not
definitions. But they may be so nearly determined in practice that they add
noise rather than signal.

So this script builds BOTH populations and carries them forward. Step 10
trains on each and compares calibration and within-group performance. The
decision is then made from the result and written into the record.

    NARROW   groups 1, 2a, 3, 4a
             term, singleton, cephalic, no uterine scar, labour occurred
    WIDE     everything classified except 2b, 4b and 9

STEP 8 — SPLITTING

Stratified on the target so the caesarean proportion is identical in every
part. Train 60 / validation 20 / test 20. The validation set tunes the
model and chooses the decision threshold. The test set is opened once, at
the very end, and never used to make a choice.

STEP 9 — IMBALANCE

Checked empirically on each restricted population, not assumed from the
national rate. This matters: the overall US rate is 32%, but groups 1 and 3
sit far below that, so the direction of imbalance in the narrow population
cannot be predicted from the headline figure.
=============================================================================
"""

import pandas as pd
import numpy as np
from pathlib import Path

SRC = "model_ready.parquet"
if not Path(SRC).exists():
    raise SystemExit(f"{SRC} not found. Run step6_encode.py first.")

d = pd.read_parquet(SRC)
FEATURES = pd.read_csv("feature_list.csv")["feature"].tolist()
TARGET = "is_cesarean"
print(f"Loaded {len(d):,} rows, {len(FEATURES)} features\n")

# =============================================================================
print("=" * 74)
print("STEP 7 — TRAINING POPULATION")
print("=" * 74)

clf = d[d["Classification_status"] == "Classified"].copy()
print(f"  Classified births              : {len(clf):,}")
print(f"  Unclassifiable, excluded       : {len(d)-len(clf):,} "
      f"({100*(len(d)-len(clf))/len(d):.2f}%)")

LEAK = ["2b", "4b", "9"]
leak_rows = clf[clf["Robson_group"].isin(LEAK)]
print(f"\n  Excluded as leakage or contaminated:")
for g in LEAK:
    sub = clf[clf["Robson_group"] == g]
    if len(sub):
        print(f"    group {g:<3} {len(sub):>9,} births   caesarean rate {100*sub[TARGET].mean():>6.2f}%")
print(f"    total    {len(leak_rows):>9,} births   "
      f"({100*leak_rows[TARGET].sum()/clf[TARGET].sum():.1f}% of ALL caesareans in the data)")
print("""
  Read that last figure. A sizeable share of every positive case in the
  dataset sits inside groups whose outcome is fixed by definition. Left in,
  they would be free marks.
""")

NARROW = ["1", "2a", "3", "4a"]
pops = {
    "narrow": clf[clf["Robson_group"].isin(NARROW)].copy(),
    "wide":   clf[~clf["Robson_group"].isin(LEAK)].copy(),
}

for name, p in pops.items():
    print(f"\n  {name.upper()} population: {len(p):,} births, "
          f"caesarean rate {100*p[TARGET].mean():.2f}%")
    comp = (p.groupby(["Robson_group_sort_order", "Robson_group"], observed=True)
              .agg(n=(TARGET, "size"), cs=(TARGET, "mean"))
              .reset_index().sort_values("Robson_group_sort_order"))
    comp["share_%"] = (100 * comp["n"] / len(p)).round(2)
    comp["cs_rate_%"] = (100 * comp["cs"]).round(2)
    print(comp[["Robson_group", "n", "share_%", "cs_rate_%"]].to_string(index=False))

# =============================================================================
print("\n" + "=" * 74)
print("STEP 8 — STRATIFIED SPLIT  (60 / 20 / 20)")
print("=" * 74)

def split(p, seed=42):
    rng = np.random.default_rng(seed)
    p = p.copy()
    p["_part"] = ""
    for y in (0, 1):                      # stratify on the target
        idx = p.index[p[TARGET] == y].to_numpy().copy()   # to_numpy() on an Index is read-only
        rng.shuffle(idx)
        n = len(idx)
        a, b = int(0.60 * n), int(0.80 * n)
        p.loc[idx[:a],  "_part"] = "train"
        p.loc[idx[a:b], "_part"] = "valid"
        p.loc[idx[b:],  "_part"] = "test"
    return p

for name in pops:
    pops[name] = split(pops[name])
    p = pops[name]
    print(f"\n  {name.upper()}")
    t = p.groupby("_part").agg(n=(TARGET, "size"), cs_rate=(TARGET, "mean"))
    t["cs_rate_%"] = (100 * t["cs_rate"]).round(3)
    print(t[["n", "cs_rate_%"]].reindex(["train", "valid", "test"]).to_string())
    spread = t["cs_rate_%"].max() - t["cs_rate_%"].min()
    print(f"    caesarean rate spread across parts: {spread:.3f} pp "
          f"{'OK' if spread < 0.5 else '<-- CHECK'}")

# =============================================================================
print("\n" + "=" * 74)
print("STEP 9 — CLASS IMBALANCE, MEASURED NOT ASSUMED")
print("=" * 74)
print("""
  The national caesarean rate is 32%. That figure does not describe either
  training population, because the groups with fixed outcomes have been
  removed. This is exactly why imbalance is checked here rather than
  inherited from the headline.
""")
for name, p in pops.items():
    tr = p[p["_part"] == "train"]
    pos = tr[TARGET].sum()
    neg = len(tr) - pos
    rate = 100 * pos / len(tr)
    spw = neg / pos
    print(f"  {name.upper():<8} train n={len(tr):>9,}   caesarean {rate:>6.2f}%   "
          f"ratio 1:{spw:.2f}   scale_pos_weight={spw:.3f}")

print("""
  How to read the ratio. Below about 1:4 the imbalance is mild and standard
  training usually copes; beyond that, class weighting earns its place and
  AUPRC becomes the metric that matters rather than ROC-AUC.

  Note what is NOT being done: no resampling before the split. SMOTE or any
  oversampling applied to the whole dataset leaks synthetic neighbours of
  test cases into training and inflates every reported number. If resampling
  is used at all it goes inside the training fold only.
""")

for name, p in pops.items():
    cols = FEATURES + [TARGET, "Robson_group", "Robson_group_sort_order", "_part"]
    p[cols].to_parquet(f"train_{name}.parquet", index=False)
    print(f"Saved train_{name}.parquet — {len(p):,} rows")

print("\nNext: step 10 — train XGBoost on both populations and compare.")

---

## Step 10 — First Training Attempt (the old version)

**Script:** `step10_train.py`

### ⚠ This is the version BEFORE the calibration fix

This script is preserved as a record of the analytical evolution. It does three things right and one thing wrong:

**Right:**
1. Fixes the split overlap (narrow is a subset of wide → births could appear in narrow-test and wide-train)
2. Head-to-head comparison on identical rows
3. Logistic regression baseline

**Wrong:**
- Uses `scale_pos_weight` to handle class imbalance. This inflates every predicted probability. The narrow model got `spw=7.561`, meaning every positive case counted 7.6× — which pushed predictions far above the true rate.

The script's automatic verdict looked only at AUPRC and missed the calibration problem entirely. Step 11b corrects this.

In [ ]:
"""
=============================================================================
STEP 10 + 11 — TRAIN, AND SETTLE THE OPEN QUESTION
=============================================================================
    pip install xgboost scikit-learn
    python step10_train.py

A DESIGN FAULT FIXED BEFORE TRAINING

Step 8 split each population independently. The narrow population is a
subset of the wide one, so the same birth could land in narrow-test and
wide-train at once. Comparing the two models on the same rows would then
flatter the wide model, because it had already seen some of them.

This script takes the WIDE split as the single reference and lets the
narrow population inherit it. A birth now has the same part in both, and
the head-to-head is clean. The narrow file's own split assignment is
ignored.

THE QUESTION BEING SETTLED

Groups 5, 6, 7, 8 and 10 are not leakage. A breech CAN deliver vaginally; a
repeat caesarean IS a decision. But their rates run so high that they may
contribute noise rather than signal, and including them lets a model earn
most of its apparent skill on cases nobody needed a model for.

The test is deliberately narrow: both models are evaluated on the SAME
rows — the held-out births in groups 1, 2a, 3 and 4a, where genuine
uncertainty lives. If the wide model does not predict those better, the
extra groups are not helping, and the narrow population is the honest
choice.

STEP 11 — A BASELINE INSTEAD OF THREE MODELS

The proposal deferred a three-family comparison. On tabular data the winner
is close to certain in advance, and every extra model is another chance to
pick the best result, which quietly inflates whatever gets reported. One
logistic regression answers the question that actually matters — are the
relationships non-linear enough to need a tree ensemble — at a fraction of
the cost. If XGBoost cannot beat it by a clear margin, that is a real
finding about the data and it gets written down.
=============================================================================
"""

import pandas as pd
import numpy as np
from pathlib import Path
import json, time

try:
    import xgboost as xgb
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.impute import SimpleImputer
    from sklearn.metrics import (roc_auc_score, average_precision_score,
                                 brier_score_loss)
except ImportError as e:
    raise SystemExit(f"Missing package: {e}\n  pip install xgboost scikit-learn")

TARGET = "is_cesarean"
NARROW = ["1", "2a", "3", "4a"]

if not Path("train_wide.parquet").exists():
    raise SystemExit("train_wide.parquet not found. Run step7_9_split.py first.")

print("Loading train_wide.parquet (the reference split) ...")
wide = pd.read_parquet("train_wide.parquet")
FEATURES = [c for c in wide.columns
            if c not in [TARGET, "Robson_group", "Robson_group_sort_order", "_part"]]
print(f"  {len(wide):,} births, {len(FEATURES)} features\n")

narrow = wide[wide["Robson_group"].isin(NARROW)].copy()

print("=" * 74)
print("SHARED SPLIT — inherited, so the comparison is clean")
print("=" * 74)
for name, p in [("wide", wide), ("narrow", narrow)]:
    t = p.groupby("_part").agg(n=(TARGET, "size"), r=(TARGET, "mean"))
    t["cs_%"] = (100 * t["r"]).round(3)
    print(f"\n  {name.upper()}")
    print(t[["n", "cs_%"]].reindex(["train", "valid", "test"]).to_string())
    sp = t["cs_%"].max() - t["cs_%"].min()
    print(f"    spread {sp:.3f} pp  {'OK' if sp < 0.5 else '<-- CHECK'}")

def parts(p):
    out = {}
    for k in ("train", "valid", "test"):
        s = p[p["_part"] == k]
        out[k] = (s[FEATURES].astype(float), s[TARGET].astype(int), s)
    return out


def train_xgb(p, label):
    D = parts(p)
    Xtr, ytr, _ = D["train"]; Xva, yva, _ = D["valid"]
    spw = (ytr == 0).sum() / (ytr == 1).sum()
    print(f"\n  Training {label}: {len(Xtr):,} rows, scale_pos_weight={spw:.3f}")
    m = xgb.XGBClassifier(
        n_estimators=600, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        min_child_weight=20, reg_lambda=1.0,
        scale_pos_weight=spw, tree_method="hist",
        eval_metric="aucpr", early_stopping_rounds=30,
        n_jobs=-1, random_state=42)
    t0 = time.time()
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    print(f"    done in {time.time()-t0:.0f}s, best iteration {m.best_iteration}")
    return m, D


def report(y, p, label, indent="    "):
    auc  = roc_auc_score(y, p)
    ap   = average_precision_score(y, p)
    br   = brier_score_loss(y, p)
    base = y.mean()
    print(f"{indent}{label:<26} ROC-AUC {auc:.4f}   AUPRC {ap:.4f} "
          f"(baseline {base:.4f})   Brier {br:.4f}")
    return {"roc_auc": auc, "auprc": ap, "brier": br, "positive_rate": float(base)}


results = {}
models = {}
print("\n" + "=" * 74)
print("STEP 10 — XGBOOST ON BOTH POPULATIONS")
print("=" * 74)
for name, p in [("narrow", narrow), ("wide", wide)]:
    m, D = train_xgb(p, name)
    models[name] = (m, D)
    Xva, yva, _ = D["valid"]
    results[f"xgb_{name}"] = report(yva, m.predict_proba(Xva)[:, 1],
                                    f"{name} on its own valid")

# ---------------------------------------------------------------------------
print("\n" + "=" * 74)
print("THE HEAD-TO-HEAD — both models, identical rows")
print("=" * 74)
print("  Held-out births in groups 1, 2a, 3 and 4a only.\n")

_, _, nva = models["narrow"][1]["valid"]
common = nva[nva["Robson_group"].isin(NARROW)]
Xc = common[FEATURES].astype(float); yc = common[TARGET].astype(int)
print(f"  n = {len(common):,}, caesarean rate {100*yc.mean():.2f}%\n")

h2h = {}
for name in ("narrow", "wide"):
    m = models[name][0]
    h2h[name] = report(yc, m.predict_proba(Xc)[:, 1], f"{name}-trained model")

d_auc = h2h["wide"]["roc_auc"] - h2h["narrow"]["roc_auc"]
d_ap  = h2h["wide"]["auprc"]   - h2h["narrow"]["auprc"]
d_br  = h2h["wide"]["brier"]   - h2h["narrow"]["brier"]
print(f"\n  wide minus narrow:  ROC-AUC {d_auc:+.4f}   AUPRC {d_ap:+.4f}   "
       f"Brier {d_br:+.4f}  (lower Brier is better)")
print(f"""
  VERDICT
  {"The wider population helps on the uncertain groups. Keep it." if d_ap > 0.005 else
    "The wider population does not improve prediction where uncertainty actually lives. Train on the narrow population and say so."}

  Either way this is now a decision made from evidence, on identical rows,
  and it goes into the record as such rather than as a preference.
""")

# ---------------------------------------------------------------------------
print("=" * 74)
print("STEP 11 — LOGISTIC REGRESSION BASELINE")
print("=" * 74)
print("""  Median imputation and standardisation, because logistic regression
  cannot take missing values the way a tree can. That handicap is part of
  the comparison, not a flaw in it: handling missingness natively is one of
  the reasons to prefer the tree.
""")
best = "wide" if d_ap > 0.005 else "narrow"
p = wide if best == "wide" else narrow
D = parts(p)
Xtr, ytr, _ = D["train"]; Xva, yva, _ = D["valid"]

sub = min(400_000, len(Xtr))
lr = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                   LogisticRegression(max_iter=1000, class_weight="balanced"))
t0 = time.time()
lr.fit(Xtr.iloc[:sub], ytr.iloc[:sub])
print(f"  fitted on {sub:,} rows in {time.time()-t0:.0f}s")
results["logreg"] = report(yva, lr.predict_proba(Xva)[:, 1], "logistic regression")
results[f"xgb_{best}_same"] = report(yva, models[best][0].predict_proba(Xva)[:, 1],
                                     f"XGBoost ({best})")

gain = results[f"xgb_{best}_same"]["auprc"] - results["logreg"]["auprc"]
print(f"""
  XGBoost gains {gain:+.4f} AUPRC over logistic regression.

  {"That is a material gain: the relationships are non-linear and the tree ensemble earns its place." if gain > 0.01 else
    "That is a small gain. The relationships are close to linear, which is itself worth reporting — and it means the interpretable model is a legitimate alternative."}
""")

models[best][0].save_model(f"xgb_{best}.json")
with open("model_metrics.json", "w") as f:
    json.dump({"chosen_population": best, "results": results,
               "head_to_head": h2h}, f, indent=2)
print(f"Saved xgb_{best}.json and model_metrics.json")
print("\nNext: step 12 — choose the decision threshold from the precision-recall curve.")

### Why a second version (`step10_train_v2.py`) was needed

The first version died silently during training — no error, no traceback, just a returned prompt. That is what a Windows out-of-memory kill looks like. XGBoost with `n_jobs=-1` makes a working copy per thread; 1.5M rows in float64 across every core exceeds laptop RAM.

Four memory fixes (none costly):
- `float32` instead of `float64` (halves the matrix)
- `n_jobs=4` instead of `-1` (caps working copies)
- `max_bin=128` instead of 256 (halves the histogram)
- `QuantileDMatrix` (builds histogram directly, releases source)

**This version still uses `scale_pos_weight`** — the calibration fix comes in Step 11b.

In [ ]:
"""
=============================================================================
STEP 10 + 11 — TRAIN  (memory-safe rewrite)
=============================================================================
    pip install xgboost scikit-learn
    python step10_train_v2.py

WHY THIS VERSION EXISTS

The first version died silently part-way through training the narrow model:
no error, no traceback, just a returned prompt. That is what a Windows
out-of-memory kill looks like. XGBoost with n_jobs=-1 makes a working copy
of the matrix per thread, and 1.5 million rows in float64 across every core
is more than a laptop has.

Four changes, none of which cost anything real:

  float32 instead of float64 ...... halves the matrix; birth weights and
                                    gestational ages do not need 15
                                    significant figures
  n_jobs=4 instead of -1 .......... caps the number of working copies
  max_bin=128 instead of 256 ...... halves the histogram, with no
                                    meaningful loss on continuous features
                                    of this resolution
  QuantileDMatrix ................. builds the histogram directly and
                                    releases the source frame, instead of
                                    holding both in memory at once

There is also a --sample switch. If the machine still cannot hold it, train
on a random subset. At this scale that costs very little: a model fitted on
500,000 births is not meaningfully worse than one fitted on 1.5 million,
because performance stopped improving with sample size long before either.

    python step10_train_v2.py --sample 500000

WHAT IS BEING DECIDED HERE
Whether the model should learn from groups 1, 2a, 3 and 4a only (narrow) or
from every group except the three excluded for leakage (wide). Both models
are scored on THE SAME held-out rows in the uncertain groups, so the
comparison is not confounded by the population being scored.
=============================================================================
"""

import pandas as pd
import numpy as np
from pathlib import Path
import json, time, gc, sys

try:
    import xgboost as xgb
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.impute import SimpleImputer
    from sklearn.metrics import (roc_auc_score, average_precision_score,
                                 brier_score_loss)
except ImportError as e:
    raise SystemExit(f"Missing package: {e}\n  pip install xgboost scikit-learn")

TARGET = "is_cesarean"
NARROW = ["1", "2a", "3", "4a"]

SAMPLE = None
if "--sample" in sys.argv:
    SAMPLE = int(sys.argv[sys.argv.index("--sample") + 1])

if not Path("train_wide.parquet").exists():
    raise SystemExit("train_wide.parquet not found. Run step7_9_split.py first.")

print("Loading train_wide.parquet ...")
wide = pd.read_parquet("train_wide.parquet")
FEATURES = [c for c in wide.columns
            if c not in [TARGET, "Robson_group", "Robson_group_sort_order", "_part"]]

# cast once, up front — this is where most of the saving comes from
for c in FEATURES:
    wide[c] = pd.to_numeric(wide[c], errors="coerce").astype("float32")
wide[TARGET] = wide[TARGET].astype("int8")
print(f"  {len(wide):,} births, {len(FEATURES)} features, float32")
print(f"  memory: {wide.memory_usage(deep=True).sum()/1e9:.2f} GB\n")

if SAMPLE and SAMPLE < len(wide):
    keep = (wide.groupby("_part", group_keys=False)
                .apply(lambda g: g.sample(n=min(len(g), int(SAMPLE*len(g)/len(wide))),
                                          random_state=42)))
    wide = keep.reset_index(drop=True)
    print(f"  SAMPLED down to {len(wide):,} births\n")

narrow_mask = wide["Robson_group"].isin(NARROW)

print("=" * 74)
print("SHARED SPLIT — the narrow population inherits the wide split")
print("=" * 74)
for name, mask in [("wide", pd.Series(True, index=wide.index)), ("narrow", narrow_mask)]:
    t = wide[mask].groupby("_part").agg(n=(TARGET, "size"), r=(TARGET, "mean"))
    t["cs_%"] = (100 * t["r"]).round(3)
    print(f"\n  {name.upper()}")
    print(t[["n", "cs_%"]].reindex(["train", "valid", "test"]).to_string())


def xy(mask, part):
    s = wide[mask & (wide["_part"] == part)]
    return s[FEATURES], s[TARGET].to_numpy(), s


def train(mask, label):
    Xtr, ytr, _ = xy(mask, "train")
    Xva, yva, _ = xy(mask, "valid")
    spw = float((ytr == 0).sum() / max((ytr == 1).sum(), 1))
    print(f"\n  Training {label}: {len(Xtr):,} rows, scale_pos_weight={spw:.3f}")

    dtr = xgb.QuantileDMatrix(Xtr, label=ytr, max_bin=128)
    dva = xgb.QuantileDMatrix(Xva, label=yva, max_bin=128, ref=dtr)
    del Xtr, Xva; gc.collect()

    params = dict(objective="binary:logistic", eval_metric="aucpr",
                  eta=0.08, max_depth=6, subsample=0.8, colsample_bytree=0.8,
                  min_child_weight=20, reg_lambda=1.0,
                  scale_pos_weight=spw, tree_method="hist",
                  max_bin=128, nthread=4, seed=42)
    t0 = time.time()
    bst = xgb.train(params, dtr, num_boost_round=300,
                    evals=[(dva, "valid")], early_stopping_rounds=30,
                    verbose_eval=50)
    print(f"    done in {time.time()-t0:.0f}s, best iteration {bst.best_iteration}")
    del dtr, dva; gc.collect()
    return bst


def predict(bst, X):
    return bst.predict(xgb.DMatrix(X), iteration_range=(0, bst.best_iteration + 1))


def report(y, p, label, indent="    "):
    m = {"roc_auc": roc_auc_score(y, p),
         "auprc": average_precision_score(y, p),
         "brier": brier_score_loss(y, p),
         "positive_rate": float(np.mean(y))}
    print(f"{indent}{label:<26} ROC-AUC {m['roc_auc']:.4f}   "
          f"AUPRC {m['auprc']:.4f} (baseline {m['positive_rate']:.4f})   "
          f"Brier {m['brier']:.4f}")
    return m


print("\n" + "=" * 74)
print("STEP 10 — XGBOOST ON BOTH POPULATIONS")
print("=" * 74)
results, models = {}, {}
for name, mask in [("narrow", narrow_mask),
                   ("wide", pd.Series(True, index=wide.index))]:
    bst = train(mask, name)
    models[name] = bst
    Xva, yva, _ = xy(mask, "valid")
    results[f"xgb_{name}"] = report(yva, predict(bst, Xva), f"{name} on its own valid")
    del Xva; gc.collect()

# ---------------------------------------------------------------------------
print("\n" + "=" * 74)
print("THE HEAD-TO-HEAD — both models, identical rows")
print("=" * 74)
print("  Held-out births in groups 1, 2a, 3 and 4a only — where the")
print("  uncertainty actually lives.\n")

Xc, yc, _ = xy(narrow_mask, "valid")
print(f"  n = {len(Xc):,}, caesarean rate {100*np.mean(yc):.2f}%\n")
h2h = {n: report(yc, predict(models[n], Xc), f"{n}-trained model")
       for n in ("narrow", "wide")}

d_auc = h2h["wide"]["roc_auc"] - h2h["narrow"]["roc_auc"]
d_ap  = h2h["wide"]["auprc"]   - h2h["narrow"]["auprc"]
d_br  = h2h["wide"]["brier"]   - h2h["narrow"]["brier"]
print(f"\n  wide minus narrow:  ROC-AUC {d_auc:+.4f}   AUPRC {d_ap:+.4f}   "
      f"Brier {d_br:+.4f}  (lower Brier is better)")

best = "wide" if d_ap > 0.005 else "narrow"
print(f"""
  VERDICT — train on the {best.upper()} population.
  {"The extra groups improve prediction where it is genuinely uncertain."
    if best == "wide" else
   "The extra groups do not improve prediction where it is genuinely uncertain, so they are contributing high-certainty cases rather than information."}

  This is now settled on identical rows, and goes into the record as a
  result rather than a preference.
""")

# ---------------------------------------------------------------------------
print("=" * 74)
print("STEP 11 — LOGISTIC REGRESSION BASELINE")
print("=" * 74)
print("""  Median imputation and standardisation, because logistic regression
  cannot take missing values the way a tree can. That handicap is part of
  the comparison rather than a flaw in it.
""")
mask = narrow_mask if best == "narrow" else pd.Series(True, index=wide.index)
Xtr, ytr, _ = xy(mask, "train")
Xva, yva, _ = xy(mask, "valid")
sub = min(300_000, len(Xtr))
lr = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                   LogisticRegression(max_iter=1000, class_weight="balanced"))
t0 = time.time()
lr.fit(Xtr.iloc[:sub], ytr[:sub])
print(f"  fitted on {sub:,} rows in {time.time()-t0:.0f}s")
results["logreg"] = report(yva, lr.predict_proba(Xva)[:, 1], "logistic regression")
results["xgb_chosen"] = report(yva, predict(models[best], Xva), f"XGBoost ({best})")

gain = results["xgb_chosen"]["auprc"] - results["logreg"]["auprc"]
print(f"""
  XGBoost gains {gain:+.4f} AUPRC over logistic regression.
  {"A material gain: the relationships are non-linear and the tree ensemble earns its place."
    if gain > 0.01 else
   "A small gain. The relationships are close to linear — worth reporting in its own right, and it means the interpretable model is a legitimate alternative."}
""")

models[best].save_model(f"xgb_{best}.json")
pd.Series(FEATURES).to_csv("model_feature_order.csv", index=False, header=["feature"])
with open("model_metrics.json", "w") as f:
    json.dump({"chosen_population": best, "n_train": int(mask.sum()),
               "sampled": SAMPLE, "results": results, "head_to_head": h2h},
              f, indent=2)
print(f"Saved xgb_{best}.json, model_feature_order.csv, model_metrics.json")
print("\nNext: step 12 — choose the decision threshold from the precision-recall curve.")

---

## Step 11b — Why the Brier Scores Differ (THE KEY CORRECTION)

**Script:** `step11b_calibration.py`

### What the head-to-head actually showed

On identical held-out rows, the two models discriminated the same: ROC-AUC 0.7985 vs 0.7986. Nothing to choose.

But Brier came out **0.1857 for narrow** and **0.1035 for wide** — the wide model 44% better. Since discrimination is identical, the entire difference is **calibration**: the narrow model ranks births correctly but the probabilities it prints are wrong.

### The hypothesis

`scale_pos_weight` was **7.561** for narrow and **2.465** for wide. Class weighting makes positives count for more, shifting every predicted probability upward. The larger the weight, the larger the distortion.

### The test: three models, one question

| Model | Population | Weighting | Purpose |
|-------|-----------|-----------|---------|
| narrow + weighting | Groups 1,2a,3,4a | spw=7.6 | The current model |
| **narrow, NO weighting** | Groups 1,2a,3,4a | spw=1.0 | **Isolates the weighting** |
| wide + weighting | All except 2b,4b,9 | spw=2.5 | The comparator |

### The result

| Model | Brier | Mean predicted | Observed |
|-------|-------|---------------|----------|
| narrow + weighting | **0.1857** | ~30% | 11.72% |
| **narrow, NO weighting** | **0.0886** | **11.70%** | 11.72% |
| wide + weighting | 0.1035 | ~17% | 11.72% |

**Removing class weighting cut Brier by more than half.** The calibration problem was the weighting, not the population. `scale_pos_weight=7.6` was inflating every probability.

**Decision:** Train on the NARROW population WITHOUT class weighting. Handle imbalance where it belongs: in the choice of metric (AUPRC) and threshold, not by distorting the model's output.

In [ ]:
"""
=============================================================================
STEP 11b — WHY THE BRIER SCORES DIFFER
=============================================================================
    python step11b_calibration.py

WHAT THE HEAD-TO-HEAD ACTUALLY SHOWED

On identical held-out rows the two models discriminated the same:
ROC-AUC 0.7985 against 0.7986, AUPRC 0.3538 against 0.3543. Nothing to
choose between them.

But Brier came out 0.1857 for narrow and 0.1035 for wide — the wide model
44% better. Brier combines discrimination and calibration. Discrimination
is identical here, so the whole of that difference is CALIBRATION: the
narrow model ranks births correctly but the probabilities it prints are
wrong.

The automatic verdict in step 10 looked only at AUPRC and therefore missed
this. The rule was too simple and is corrected here.

WHY CALIBRATION MATTERS MORE THAN DISCRIMINATION IN THIS PROJECT

Most clinical prediction papers care about ranking: who is at higher risk.
This project does something different. It sums predicted probabilities to
produce an EXPECTED caesarean rate, and subtracts that from the observed
rate to estimate excess. A model whose probabilities are inflated gives an
inflated expected rate and therefore an understated gap. Ranking correctly
is not enough — the numbers themselves have to be right.

THE HYPOTHESIS
scale_pos_weight was 7.561 for the narrow model and 2.465 for the wide one.
Class weighting corrects imbalance by making positives count for more,
which shifts every predicted probability upward. The larger the weight, the
larger the distortion. If that is the cause, the population is innocent and
the fix is to drop the weighting rather than to change who the model learns
from.

Three models, one question:
    narrow WITH weighting      the current model
    narrow WITHOUT weighting   isolates the weighting
    wide WITH weighting        the comparator
All scored on the same held-out rows.
=============================================================================
"""

import pandas as pd
import numpy as np
from pathlib import Path
import gc, json

import xgboost as xgb
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             brier_score_loss)

TARGET = "is_cesarean"
NARROW = ["1", "2a", "3", "4a"]

wide = pd.read_parquet("train_wide.parquet")
FEATURES = [c for c in wide.columns
            if c not in [TARGET, "Robson_group", "Robson_group_sort_order", "_part"]]
for c in FEATURES:
    wide[c] = pd.to_numeric(wide[c], errors="coerce").astype("float32")
wide[TARGET] = wide[TARGET].astype("int8")
nmask = wide["Robson_group"].isin(NARROW)
print(f"Loaded {len(wide):,} births\n")


def xy(mask, part):
    s = wide[mask & (wide["_part"] == part)]
    return s[FEATURES], s[TARGET].to_numpy()


def fit(mask, weighted, label):
    Xtr, ytr = xy(mask, "train")
    Xva, yva = xy(mask, "valid")
    spw = float((ytr == 0).sum() / max((ytr == 1).sum(), 1)) if weighted else 1.0
    print(f"  {label:<28} n={len(Xtr):>9,}  scale_pos_weight={spw:.3f}")
    dtr = xgb.QuantileDMatrix(Xtr, label=ytr, max_bin=128)
    dva = xgb.QuantileDMatrix(Xva, label=yva, max_bin=128, ref=dtr)
    del Xtr, Xva; gc.collect()
    p = dict(objective="binary:logistic", eval_metric="aucpr", eta=0.08,
             max_depth=6, subsample=0.8, colsample_bytree=0.8,
             min_child_weight=20, reg_lambda=1.0, scale_pos_weight=spw,
             tree_method="hist", max_bin=128, nthread=4, seed=42)
    b = xgb.train(p, dtr, num_boost_round=300, evals=[(dva, "valid")],
                  early_stopping_rounds=30, verbose_eval=False)
    del dtr, dva; gc.collect()
    return b


def predict(b, X):
    return b.predict(xgb.DMatrix(X), iteration_range=(0, b.best_iteration + 1))


print("Training three models ...")
models = {
    "narrow + weighting":    fit(nmask, True,  "narrow + weighting"),
    "narrow, NO weighting":  fit(nmask, False, "narrow, NO weighting"),
    "wide + weighting":      fit(pd.Series(True, index=wide.index), True,
                                 "wide + weighting"),
}

# ---- score all three on the SAME rows -------------------------------------
Xc, yc = xy(nmask, "valid")
print("\n" + "=" * 78)
print("ALL THREE ON THE SAME HELD-OUT ROWS")
print(f"n = {len(Xc):,}, observed caesarean rate {100*yc.mean():.2f}%")
print("=" * 78)
print(f"\n{'model':<24} {'ROC-AUC':>9} {'AUPRC':>8} {'Brier':>8} "
      f"{'mean predicted':>15} {'observed':>10}")
print("-" * 78)

preds, rows = {}, {}
for name, b in models.items():
    p = predict(b, Xc)
    preds[name] = p
    rows[name] = {"roc_auc": float(roc_auc_score(yc, p)),
                  "auprc": float(average_precision_score(yc, p)),
                  "brier": float(brier_score_loss(yc, p)),
                  "mean_pred": float(p.mean())}
    print(f"{name:<24} {rows[name]['roc_auc']:>9.4f} {rows[name]['auprc']:>8.4f} "
          f"{rows[name]['brier']:>8.4f} {100*p.mean():>14.2f}% {100*yc.mean():>9.2f}%")

print("""
  The last two columns are the calibration test in its simplest form. If a
  model is calibrated, the average probability it predicts across a
  population equals the rate actually observed in it. A model predicting a
  mean of 30% where 11.7% of births were caesarean is not slightly off — it
  would overstate Egypt's expected rate by a factor of two and shrink the
  estimated gap accordingly.
""")

# ---- calibration curve ----------------------------------------------------
print("=" * 78)
print("CALIBRATION CURVE — predicted against observed, by decile of risk")
print("=" * 78)
for name, p in preds.items():
    q = pd.qcut(p, 10, labels=False, duplicates="drop")
    t = pd.DataFrame({"bin": q, "pred": p, "obs": yc}).groupby("bin").agg(
        n=("obs", "size"), predicted=("pred", "mean"), observed=("obs", "mean"))
    t["predicted_%"] = (100 * t["predicted"]).round(2)
    t["observed_%"] = (100 * t["observed"]).round(2)
    t["gap_pp"] = (t["predicted_%"] - t["observed_%"]).round(2)
    print(f"\n  {name}")
    print(t[["n", "predicted_%", "observed_%", "gap_pp"]].to_string())
    print(f"    mean absolute gap: {t['gap_pp'].abs().mean():.2f} pp")

# ---- verdict --------------------------------------------------------------
nb_w  = rows["narrow + weighting"]["brier"]
nb_nw = rows["narrow, NO weighting"]["brier"]
wb    = rows["wide + weighting"]["brier"]

print("\n" + "=" * 78)
print("VERDICT")
print("=" * 78)
if nb_nw < nb_w * 0.75:
    print(f"""
  Removing class weighting cut the narrow model's Brier from {nb_w:.4f} to
  {nb_nw:.4f}. The calibration problem was the weighting, not the
  population. scale_pos_weight of 7.6 was inflating every probability.

  DECISION: train on the NARROW population WITHOUT class weighting.
  Discrimination was already identical between populations, so the narrow
  set — the births where a clinician faces genuine uncertainty — is the
  honest one to learn from, and dropping the weight fixes the probabilities.

  Imbalance is then handled where it belongs: in the choice of metric
  (AUPRC rather than accuracy) and in the decision threshold, not by
  distorting the model's output.
""")
    choice = "narrow_unweighted"
else:
    print(f"""
  Removing the weighting moved Brier from {nb_w:.4f} to {nb_nw:.4f}, which
  does not account for the gap against the wide model at {wb:.4f}. The
  weighting is not the whole story and the population itself is doing work.

  DECISION: carry both forward to step 12 and choose on calibrated
  performance after threshold selection.
""")
    choice = "undecided"

best = models["narrow, NO weighting"] if choice == "narrow_unweighted" \
       else models["wide + weighting"]
best.save_model("xgb_final.json")
with open("calibration_comparison.json", "w") as f:
    json.dump({"choice": choice, "results": rows,
               "observed_rate": float(yc.mean())}, f, indent=2)
print("Saved xgb_final.json and calibration_comparison.json")
print("\nNext: step 12 — decision threshold from the precision-recall curve.")

---

## Step 12 — The Decision Threshold

**Script:** `step12_threshold.py`

### Where the threshold IS and IS NOT used

The Egypt expected-rate calculation **sums predicted probabilities** — no threshold. Calibration is what matters there, and it is now documented.

A threshold is only needed for: reporting precision/recall/F1, and any individual-level use.

### A correction to the proposal

The proposal committed to F2 (recall-weighted). That is withdrawn.

**Why:** This model predicts whether a caesarean will be **performed**, not whether one is **needed**. A false positive is not harmless over-caution — it tells a clinician a woman is high-risk, in a country where 82.5% of CS decisions are taken before labour begins. A recall-maximising threshold does not catch more genuine need; it manufactures more prediction, and prediction here is capable of **causing the outcome it predicts**.

### Recommended approach

| Use case | Threshold | Reason |
|----------|-----------|--------|
| Egypt expected-rate | None | Sum probabilities directly |
| Quality monitoring / audit | Volume-matched | Predicts the same count as observed |
| Individual counselling | F0.5 (precision-weighted) | Framed as probability, never as reason to book |

In [ ]:
"""
=============================================================================
STEP 12 — THE DECISION THRESHOLD
=============================================================================
    python step12_threshold.py

FIRST, WHERE THE THRESHOLD IS AND IS NOT USED

The project's central calculation — the expected caesarean rate for Egypt —
sums predicted PROBABILITIES. It never converts them to yes/no, so it uses
no threshold at all. That calculation depends on calibration, which step
11b established at a mean absolute error of 0.11 percentage points across
risk deciles.

A threshold is needed only for two secondary things: reporting precision,
recall, F1 and a confusion matrix, and any use of the model as an
individual-level flag.

A CORRECTION TO WHAT THE PROPOSAL SAYS

The proposal states the threshold will maximise F2, weighting recall on the
grounds that failing to anticipate a needed caesarean is worse than the
reverse. That reasoning is correct for a diagnostic test. It is not correct
here, and the difference matters.

This model does not predict whether a caesarean is NEEDED. It predicts
whether one will be PERFORMED — the project says so in its own methodology,
and that distinction is what makes the Egypt gap meaningful at all. So a
false positive is not a harmless over-caution. It is a tool telling an
Egyptian clinician that a woman is at high risk of caesarean, in a country
where 82.5% of caesarean decisions are already taken before labour begins
and the national rate is around 80%. A recall-maximising threshold in that
setting does not catch more genuine need; it manufactures more prediction,
and prediction here is capable of causing the outcome it predicts.

The project already holds this principle, from ACOG guidance on the VBAC
score: a predicted success probability must never become a barrier to
trial of labour. The same logic applies in reverse to this model. A
caesarean-risk score must never become a justification for booking one.

WHAT THIS SCRIPT DOES

Reports the full precision-recall curve and the operating points that
maximise F0.5, F1 and F2, plus the threshold at which predicted volume
matches observed volume. It recommends none of them silently: the choice
depends on what the number is used for, and each is shown with the
consequence spelled out in cases.
=============================================================================
"""

import pandas as pd
import numpy as np
from pathlib import Path
import json

import xgboost as xgb
from sklearn.metrics import precision_recall_curve, roc_auc_score, average_precision_score

TARGET = "is_cesarean"
NARROW = ["1", "2a", "3", "4a"]

if not Path("xgb_final.json").exists():
    raise SystemExit("xgb_final.json not found. Run step11b_calibration.py first.")

wide = pd.read_parquet("train_wide.parquet")
FEATURES = [c for c in wide.columns
            if c not in [TARGET, "Robson_group", "Robson_group_sort_order", "_part"]]
for c in FEATURES:
    wide[c] = pd.to_numeric(wide[c], errors="coerce").astype("float32")

val = wide[wide["Robson_group"].isin(NARROW) & (wide["_part"] == "valid")]
X, y = val[FEATURES], val[TARGET].to_numpy().astype(int)

bst = xgb.Booster(); bst.load_model("xgb_final.json")
p = bst.predict(xgb.DMatrix(X))

print(f"Validation set: {len(y):,} births, observed caesarean rate {100*y.mean():.2f}%")
print(f"Mean predicted probability: {100*p.mean():.2f}%")
print(f"ROC-AUC {roc_auc_score(y,p):.4f}   AUPRC {average_precision_score(y,p):.4f}\n")

prec, rec, thr = precision_recall_curve(y, p)
prec, rec = prec[:-1], rec[:-1]

def fbeta(pr, rc, b):
    return np.where((pr + rc) > 0, (1+b*b)*pr*rc / np.maximum(b*b*pr + rc, 1e-12), 0)

points = {}
for name, b in [("F0.5 (precision-weighted)", 0.5),
                ("F1 (balanced)", 1.0),
                ("F2 (recall-weighted)", 2.0)]:
    f = fbeta(prec, rec, b)
    i = int(np.argmax(f))
    points[name] = (float(thr[i]), float(prec[i]), float(rec[i]), float(f[i]))

# threshold where predicted volume equals observed volume
target_n = int(y.sum())
order = np.sort(p)[::-1]
vol_thr = float(order[target_n - 1])
pred = (p >= vol_thr).astype(int)
tp = int(((pred == 1) & (y == 1)).sum()); fp = int(((pred == 1) & (y == 0)).sum())
fn = int(((pred == 0) & (y == 1)).sum())
points["Volume-matched"] = (vol_thr, tp/max(tp+fp,1), tp/max(tp+fn,1),
                            float(fbeta(np.array([tp/max(tp+fp,1)]),
                                        np.array([tp/max(tp+fn,1)]), 1.0)[0]))

print("=" * 88)
print("OPERATING POINTS")
print("=" * 88)
print(f"{'criterion':<28} {'threshold':>10} {'precision':>10} {'recall':>8} "
      f"{'flagged':>10} {'false alarms':>13}")
print("-" * 88)
for name, (t, pr, rc, f) in points.items():
    flag = int((p >= t).sum())
    fp_n = int(((p >= t) & (y == 0)).sum())
    print(f"{name:<28} {t:>10.3f} {pr:>10.3f} {rc:>8.3f} {flag:>10,} {fp_n:>13,}")

print(f"\n  For scale: {int(y.sum()):,} caesareans actually occurred among "
      f"{len(y):,} births.")

print("""
=========================================================================
READ THE LAST TWO COLUMNS TOGETHER
=========================================================================
  "Flagged" is how many women the tool would label as high risk.
  "False alarms" is how many of those did not have a caesarean.

  The recall-weighted point catches more true cases, and the price is
  visible above. In a health system where a prediction can become the
  reason for the intervention, those false alarms are not a statistical
  cost carried by the analyst. They are carried by women.

  RECOMMENDATION
  Do not adopt a single threshold as a project-wide default, and do not
  adopt F2. Report the curve, and state the operating point separately for
  each use:

    Egypt expected-rate calculation ... no threshold. Probabilities are
                                        summed directly. Calibration is
                                        what matters and it is documented.
    Quality-monitoring / audit ........ volume-matched. Predicting the
                                        same number of caesareans that
                                        occurred keeps aggregate
                                        comparisons honest.
    Individual counselling ............ F0.5 or higher precision, with the
                                        output framed as a probability
                                        rather than a recommendation, and
                                        never as a reason to book.

  The proposal's F2 commitment is withdrawn, with this reasoning recorded.
""")

# ---- the curve, at a readable resolution ----------------------------------
print("=" * 88)
print("PRECISION-RECALL CURVE")
print("=" * 88)
print(f"{'threshold':>10} {'precision':>10} {'recall':>8} {'F1':>7} "
      f"{'flagged':>10} {'% of births':>12}")
print("-" * 88)
for t in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70]:
    pred = (p >= t)
    tp = int((pred & (y == 1)).sum()); fp = int((pred & (y == 0)).sum())
    fn = int((~pred & (y == 1)).sum())
    pr = tp/max(tp+fp, 1); rc = tp/max(tp+fn, 1)
    f1 = 2*pr*rc/max(pr+rc, 1e-12)
    print(f"{t:>10.2f} {pr:>10.3f} {rc:>8.3f} {f1:>7.3f} {int(pred.sum()):>10,} "
          f"{100*pred.mean():>11.2f}%")

out = {"observed_rate": float(y.mean()), "mean_predicted": float(p.mean()),
       "roc_auc": float(roc_auc_score(y, p)),
       "auprc": float(average_precision_score(y, p)),
       "operating_points": {k: {"threshold": v[0], "precision": v[1],
                                "recall": v[2], "f": v[3]} for k, v in points.items()},
       "recommended": {"egypt_calculation": None,
                       "quality_monitoring": vol_thr,
                       "individual_counselling": points["F0.5 (precision-weighted)"][0]}}
with open("threshold_analysis.json", "w") as f:
    json.dump(out, f, indent=2)
print("\nSaved threshold_analysis.json")
print("\nNext: step 13 — full evaluation on the held-out test set, opened once.")

---

## Step 13 — Final Evaluation on the Held-Out Test Set

**Script:** `step13_evaluate.py`

### The rule that makes this number worth anything

Every figure so far came from the validation set, which was used to choose the model, drop class weighting, and examine thresholds. Numbers from a set you have made decisions on are optimistic by construction.

**The test set has never been read.** No choice was made with reference to it. It is opened **once**, here, and whatever it says is what gets written down. If the result is worse than hoped, the response is to report it — not to adjust and re-run.

### What to look for

| Metric | Validation | Test (expected) |
|--------|-----------|-----------------|
| ROC-AUC | 0.7985 | Should be close |
| AUPRC | 0.3535 | Should be close |
| Brier | 0.0886 | Should be close |
| Mean predicted | 11.70% | Should match observed |

Close agreement = clean split, nothing leaked. A large drop = model was tuned to validation, not the problem.

### Two things the by-group table reveals

1. **Within-group discrimination is lower than the headline** — much of the overall ROC-AUC comes from separating groups (Group 3 at 2.5% vs Group 2a at 29%). The honest description is the within-group range.

2. **Overall calibration is errors cancelling out** — the model over-predicts Group 1 and under-predicts Group 2a. They offset on this population. On Egypt's very different distribution, they would not. This is why the Egypt expected rate uses **observed Robson rates**, not this model's predictions.

In [ ]:
"""
=============================================================================
STEP 13 — FINAL EVALUATION ON THE HELD-OUT TEST SET
=============================================================================
    python step13_evaluate.py

THE RULE THAT MAKES THIS NUMBER WORTH ANYTHING

Every figure so far came from the validation set, which was used to choose
the model, drop class weighting and examine thresholds. Numbers from a set
you have made decisions on are optimistic by construction.

The test set has never been read. No choice in this project has been made
with reference to it. It is opened once, here, and whatever it says is what
gets written down.

If the result is worse than hoped, the correct response is to report it —
not to adjust the model and run again. Doing that would turn the test set
into a second validation set, and every subsequent number would be
flattering in a way nobody could detect afterwards.

A USEFUL SELF-CHECK
Validation gave ROC-AUC 0.7985, AUPRC 0.3535, Brier 0.0886, and a mean
predicted probability of 11.70% against 11.72% observed. If the test set
lands close to those, nothing has leaked between the splits. A large gap
would be the warning sign, and it is worth looking at that comparison
before looking at anything else.
=============================================================================
"""

import pandas as pd
import numpy as np
from pathlib import Path
import json

import xgboost as xgb
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             brier_score_loss, precision_recall_fscore_support,
                             confusion_matrix)

TARGET = "is_cesarean"
NARROW = ["1", "2a", "3", "4a"]

for f in ["xgb_final.json", "train_wide.parquet", "threshold_analysis.json"]:
    if not Path(f).exists():
        raise SystemExit(f"{f} not found — run the earlier steps first.")

thr_cfg = json.load(open("threshold_analysis.json"))
VOL_THR = thr_cfg["recommended"]["quality_monitoring"]
PREC_THR = thr_cfg["recommended"]["individual_counselling"]

wide = pd.read_parquet("train_wide.parquet")
FEATURES = [c for c in wide.columns
            if c not in [TARGET, "Robson_group", "Robson_group_sort_order", "_part"]]
for c in FEATURES:
    wide[c] = pd.to_numeric(wide[c], errors="coerce").astype("float32")

te = wide[wide["Robson_group"].isin(NARROW) & (wide["_part"] == "test")].copy()
X, y = te[FEATURES], te[TARGET].to_numpy().astype(int)

bst = xgb.Booster(); bst.load_model("xgb_final.json")
p = bst.predict(xgb.DMatrix(X))
te["pred"] = p

print("=" * 76)
print("TEST SET — opened once")
print("=" * 76)
print(f"  n = {len(y):,} births, never seen by the model")
print(f"  observed caesarean rate {100*y.mean():.2f}%\n")

auc  = roc_auc_score(y, p)
ap   = average_precision_score(y, p)
brier= brier_score_loss(y, p)

VALID = {"roc_auc": 0.7985, "auprc": 0.3535, "brier": 0.0886, "mean_pred": 0.1170}
print(f"{'metric':<26} {'validation':>12} {'TEST':>12} {'difference':>12}")
print("-" * 66)
for k, v, t in [("ROC-AUC", VALID['roc_auc'], auc),
                ("AUPRC", VALID['auprc'], ap),
                ("Brier (lower better)", VALID['brier'], brier),
                ("mean predicted", VALID['mean_pred'], p.mean())]:
    print(f"{k:<26} {v:>12.4f} {t:>12.4f} {t-v:>+12.4f}")
print(f"{'observed rate':<26} {'':>12} {y.mean():>12.4f}")
print("""
  Close agreement between the two columns means the split was clean and
  nothing leaked. A large drop on the test set would mean the model had
  been tuned to the validation set rather than to the problem.
""")

# ---- calibration ----------------------------------------------------------
print("=" * 76)
print("CALIBRATION ON UNSEEN DATA")
print("=" * 76)
q = pd.qcut(p, 10, labels=False, duplicates="drop")
cal = pd.DataFrame({"bin": q, "pred": p, "obs": y}).groupby("bin").agg(
    n=("obs", "size"), predicted=("pred", "mean"), observed=("obs", "mean"))
cal["predicted_%"] = (100*cal["predicted"]).round(2)
cal["observed_%"]  = (100*cal["observed"]).round(2)
cal["gap_pp"]      = (cal["predicted_%"] - cal["observed_%"]).round(2)
print(cal[["n", "predicted_%", "observed_%", "gap_pp"]].to_string())
mae = cal["gap_pp"].abs().mean()
print(f"\n  mean absolute calibration error: {mae:.2f} pp")
print("""
  This is the number the Egypt calculation depends on. The expected rate is
  a sum of these probabilities, so an error here transfers directly into the
  estimated gap. Discrimination affects who is ranked highest; calibration
  affects whether the total is right.
""")

# ---- operating points -----------------------------------------------------
print("=" * 76)
print("PERFORMANCE AT THE TWO RECOMMENDED OPERATING POINTS")
print("=" * 76)
for name, t in [("Quality monitoring (volume-matched)", VOL_THR),
                ("Individual counselling (precision-weighted)", PREC_THR)]:
    pred = (p >= t).astype(int)
    pr, rc, f1, _ = precision_recall_fscore_support(y, pred, average="binary",
                                                    zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    print(f"\n  {name}   threshold {t:.3f}")
    print(f"    precision {pr:.3f}   recall {rc:.3f}   F1 {f1:.3f}")
    print(f"    confusion matrix")
    print(f"                      predicted no    predicted caesarean")
    print(f"      actual no       {tn:>12,}   {fp:>20,}")
    print(f"      actual caesarean{fn:>12,}   {tp:>20,}")
    print(f"    of {int(pred.sum()):,} women flagged, {fp:,} did not have a caesarean")

print("""
  Accuracy is deliberately not reported. On a population that is 88%
  vaginal, a model predicting "no caesarean" for everyone scores 88% and
  knows nothing. That is why the pre-registered metrics were precision,
  recall, F1, ROC-AUC and AUPRC.
""")

# ---- by Robson group ------------------------------------------------------
print("=" * 76)
print("WHERE THE MODEL IS STRONG AND WHERE IT IS WEAK")
print("=" * 76)
rows = []
for g, sub in te.groupby("Robson_group", observed=True):
    yy = sub[TARGET].to_numpy().astype(int); pp = sub["pred"].to_numpy()
    if len(np.unique(yy)) < 2:
        continue
    rows.append({"group": str(g), "n": int(len(sub)),
                 "observed_%": round(float(100*yy.mean()), 2),
                 "predicted_%": round(float(100*pp.mean()), 2),
                 "gap_pp": round(float(100*(pp.mean()-yy.mean())), 2),
                 "roc_auc": round(float(roc_auc_score(yy, pp)), 4),
                 "auprc": round(float(average_precision_score(yy, pp)), 4)})
r = pd.DataFrame(rows).sort_values("group")
print(r.to_string(index=False))
print("""
  Reporting one average across groups would hide this. A model that works
  well in group 3 and poorly in group 2a is not the same instrument as one
  that performs evenly, and a reader is entitled to know which it is.
""")

within = [r["roc_auc"] for r in rows]
print("=" * 76)
print("TWO THINGS THE GROUP TABLE REVEALS")
print("=" * 76)
print(f"""
  1. WITHIN-GROUP DISCRIMINATION IS {min(within):.3f} TO {max(within):.3f},
     not {auc:.4f}.

     Much of the headline figure comes from separating the groups from one
     another — group 3 runs at 2.5% and group 2a at 29%, so telling them
     apart is easy and inflates the average. Hold the group constant and the
     model is markedly weaker. The honest description of its skill is the
     within-group range, and that is what should be reported.

  2. THE OVERALL CALIBRATION OF {mae:.2f} pp IS ERRORS CANCELLING OUT.

     The model over-predicts group 1 and under-predicts group 2a. Summed
     across a population with this group mix they offset almost exactly. On
     a population with a DIFFERENT mix they would not, and the error would
     no longer be invisible.

     This is why the Egypt expected rate is computed from OBSERVED
     group-specific rates in the Robson table, not by applying this model to
     an Egyptian case mix. Egypt's distribution is nothing like this one —
     group 5 is three times larger and group 1 far smaller — so a model
     carrying opposite biases in different groups would produce a distorted
     expected rate and therefore a distorted gap.

     The model's role is explanatory: SHAP, and the demonstration that
     clinical variables account for part of the variation and not all of it.
     It is not in the critical path of the project's central estimate.
""")

out = {"n_test": int(len(y)), "observed_rate": float(y.mean()),
       "roc_auc": float(auc), "auprc": float(ap), "brier": float(brier),
       "mean_predicted": float(p.mean()),
       "calibration_mae_pp": float(mae),
       "validation_comparison": VALID,
       "by_group": rows,
       "within_group_auc_range": [float(min(within)), float(max(within))]}
with open("test_results.json", "w") as f:
    json.dump(out, f, indent=2)
r.to_csv("test_results_by_group.csv", index=False)
cal.to_csv("test_calibration_curve.csv")
print("Saved test_results.json, test_results_by_group.csv, test_calibration_curve.csv")
print("\nNext: step 14 — SHAP.")

---

## Summary

### Pipeline
```
step4_5_clean → step6_encode → step7_9_split → step10_train (old)
                                                     ↓
                                              step10_train_v2 (memory fix)
                                                     ↓
                                              step11b_calibration (THE FIX)
                                                     ↓
                                              step12_threshold → step13_evaluate
```

### Corrections documented
| What | Old | Corrected | Why |
|------|-----|-----------|-----|
| Class weighting | scale_pos_weight=7.6 | **Removed (spw=1.0)** | Inflated every probability, corrupted calibration |
| Brier score | 0.1857 | **0.0886** | Direct consequence of removing weighting |
| F2 threshold | Committed in proposal | **Withdrawn** | Model predicts "performed" not "needed" — F2 manufactures prediction |
| Population | Undecided | **Narrow** (1, 2a, 3, 4a) | Discrimination identical; narrow is the honest choice |

### Final model metrics (test set, 486,820 births, opened once)
| Metric | Value | Note |
|--------|-------|------|
| ROC-AUC | **0.7991** | Test set; validation was 0.7985 — clean split confirmed |
| AUPRC | 0.3542 | Baseline 0.1172 |
| Brier | **0.0886** | Calibrated: mean predicted 11.70% vs observed 11.72% |
| Within-group ROC-AUC | **0.678–0.705** | The range that should be reported |

### The model's role in the project

The model is **NOT** in the critical path of the Egypt gap estimate. That uses observed Robson group-specific rates. The model's role is:
1. **SHAP** — explaining what drives CS prediction (Notebook 5)
2. **Demonstrating** that clinical variables account for part of the variation but not all
3. **Practice vs risk** — showing that practice variables explain residual variation

### Next notebooks
- **Notebook 5** → SHAP interpretation & temporal validation
- **Notebook 6** → Export & packaging

---
*End of Notebook 4*